# subprocess

> Subprocess execution utilities for running shell commands

In [ ]:
#| default_exp utils.subprocess

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional

import shlex
import subprocess
from pathlib import Path

from nbdev_mcp.utils.paths import discover_env_name

## Output Utilities

In [ ]:
#| export
def tail(s: str | None, limit: int = 40000) -> str:
    """Return the tail of a long string, truncating if it exceeds limit.
    
    Parameters
    ----------
    s : str or None
        String to potentially truncate.
    limit : int
        Maximum length before truncation (default 40000).
    
    Returns
    -------
    str
        Original string if under limit, otherwise truncated with indicator.
    """
    if not s:
        return ""
    return s if len(s) <= limit else f"...[truncated {len(s)-limit} chars]...\n" + s[-limit:]

In [ ]:
#| export
def ok(errcode: int) -> bool:
    """Return True if the given error code indicates success (zero).
    
    Parameters
    ----------
    errcode : int
        Process return code.
    
    Returns
    -------
    bool
        True if errcode is zero.
    """
    return int(errcode) == 0

## Executable Discovery

In [ ]:
#| export
def which(names: List[str]) -> Optional[str]:
    """Find the first executable in the given list that exists in PATH.
    
    Parameters
    ----------
    names : List[str]
        List of executable names to search for.
    
    Returns
    -------
    str or None
        The first found executable name, or None if none found.
    
    Examples
    --------
    >>> which(['mamba', 'conda', 'pip'])
    'mamba'  # if mamba is installed
    """
    for n in names:
        try:
            subprocess.run([n, "-V"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            return n
        except Exception:
            continue
    return None

## Environment Wrapping

In [ ]:
#| export
def wrap_with_env(
    cmd: List[str],
    project: Path,
    use_env: bool = True,
    pkg_mgrs: List[str] | None = None,
) -> List[str]:
    """Wrap a command to run inside the project's conda/mamba environment.
    
    If use_env is True and the project has an associated conda/mamba env,
    prefix the command with 'mamba/conda run -n <env_name>'.
    
    Parameters
    ----------
    cmd : List[str]
        Command to wrap.
    project : Path
        Path to the nbdev project root.
    use_env : bool
        If True, attempt to wrap with environment activation.
    pkg_mgrs : List[str] or None
        Package managers to try, in order. Defaults to ['mamba', 'conda', 'micromamba'].
    
    Returns
    -------
    List[str]
        The command, possibly prefixed with environment activation.
    """
    if not use_env:
        return cmd
    
    pkg_mgrs = pkg_mgrs or ["mamba", "conda", "micromamba"]
    exe = which(pkg_mgrs)
    
    env_name = discover_env_name(project)
    if env_name and exe:
        return [exe, "run", "-n", env_name] + cmd
    return cmd  # fallback: run in the current server environment

## Command Execution

In [ ]:
#| export
def run(cmd: List[str], cwd: Path) -> Dict[str, Any]:
    """Execute a subprocess command and capture its output.
    
    Parameters
    ----------
    cmd : List[str]
        Command and arguments to execute.
    cwd : Path
        Working directory for the command.
    
    Returns
    -------
    Dict[str, Any]
        Dictionary with keys:
        
        - cmd: The command as a quoted string
        - cwd: Working directory path
        - returncode: Process return code
        - stdout: Standard output (truncated if too long)
        - stderr: Standard error (truncated if too long)
        - ok: True if returncode is zero
    
    Examples
    --------
    >>> result = run(['echo', 'hello'], Path('.'))
    >>> result['ok']
    True
    >>> result['stdout'].strip()
    'hello'
    """
    proc = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    return {
        "cmd": " ".join(shlex.quote(x) for x in cmd),
        "cwd": str(cwd),
        "returncode": proc.returncode,
        "stdout": tail(proc.stdout),
        "stderr": tail(proc.stderr),
        "ok": ok(proc.returncode),
    }

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()